# **Automated Video Transcription and Preservation Workflow**

This script automates the transcription of video files and their preservation using [WhisperX](https://github.com/m-bain/whisperX) and [pyPreservica](https://github.com/carj/pyPreservica), integrating Google Drive for file management.

## **Workflow Overview**

1. **Install Tools**: The script installs WhisperX for speech-to-text transcription and `ffmpeg` for multimedia handling.

2. **Google Drive Integration**: It mounts Google Drive to access and manage media files.

3. **Directory Setup**: The script sets up directories for videos to be processed and stores the processed outputs.

4. **Video Transcription**: Videos are transcribed using WhisperX, generating `.srt` subtitle files, which are then moved to the processed folder.

5. **Preservica Upload**: After processing, the script uploads the videos and subtitles to Preservica for secure digital preservation.

## Mount Google Drive

In [1]:
# This step is necessary to access files stored on your Google Drive, especially if you are using Google Colab.
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## T4 (GPU Access)

In [2]:
# Set this to True if connected to a T4 runtime, else set to False if connected to CPU runtime
gpu_access = True

# WhisperX

## Install necessary packages

In [ ]:
# THIS ENTIRE CELL EXAMPLE IS FROM HERE: https://github.com/m-bain/whisperX/issues/1087

!pip uninstall torch torchvision torchaudio -y

# Workaround from: https://github.com/m-bain/whisperX/issues/1027#issuecomment-2627525081
!pip install torch==2.5.1 torchvision==0.20.1 torchaudio==2.5.1 --index-url https://download.pytorch.org/whl/cu121

# WhisperX-related packages:
!pip install ctranslate2==4.4.0
!pip install faster-whisper==1.1.0
# !pip install git+https://github.com/m-bain/whisperx.git
!pip install whisperx==3.3.1

!apt-get update
!apt-get install libcudnn8=8.9.2.26-1+cuda12.1
!apt-get install libcudnn8-dev=8.9.2.26-1+cuda12.1

!python -c "import torch; torch.backends.cuda.matmul.allow_tf32 = True; torch.backends.cudnn.allow_tf32 = True"

print('WhisperX installation complete!')


Usage:   
  pip3 uninstall [options] <package> ...
  pip3 uninstall [options] -r <requirements file> ...

no such option: -g
Looking in indexes: https://download.pytorch.org/whl/cu121
Hit:1 https://cli.github.com/packages stable InRelease
Hit:2 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Hit:3 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:4 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:5 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:6 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:7 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:8 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:10 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:11 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Reading package lists... Done
W: Skipping acquir

## Import required libraries

In [4]:
import os
import shutil

## Define paths for the directories

In [5]:
# Define paths for the directories
base_folder = '/content/drive/My Drive/Media for transcription'
to_process_folder = os.path.join(base_folder, 'To be processed')
processed_folder = os.path.join(base_folder, 'Processed')
processed_and_ingested_folder = os.path.join(base_folder, 'Processed and ingested')

## Create the necessary directories if they don't exist

In [6]:
# Create the necessary directories if they don't exist
os.makedirs(to_process_folder, exist_ok=True)
os.makedirs(processed_folder, exist_ok=True)
os.makedirs(processed_and_ingested_folder, exist_ok=True)

## Loop through all files in the 'To be processed' folder

In [ ]:
# PARALLELIZED VERSION: Process multiple videos concurrently
# Based on your resources (9.4 GB free GPU RAM), 2 parallel instances is safe
# Each WhisperX instance uses ~2-4 GB GPU RAM

from concurrent.futures import ThreadPoolExecutor, as_completed
import subprocess
import threading
import time
from datetime import datetime

# Configure number of parallel workers
# With ~14.7 GB free GPU RAM available, you can safely run:
# - 2 workers: Very safe (uses ~4-8 GB total)
# - 3 workers: Should work well (uses ~6-12 GB total) - RECOMMENDED
# - 4 workers: Possible but risky, monitor for OOM errors (uses ~8-16 GB total)
# 
# ⚠️  WARNING: If GPU RAM spikes to 15 GB (100%), reduce NUM_WORKERS immediately!
NUM_WORKERS = 3

def process_video(filename):
    """Process a single video file with WhisperX"""
    worker_id = threading.current_thread().name
    
    if not filename.endswith(('.mp4', '.mkv', '.avi', '.mov', '.flv', '.wmv', '.m4v')):
        return {'filename': filename, 'status': 'skipped', 'reason': 'not a video file'}
    
    input_path = os.path.join(to_process_folder, filename)
    output_folder = processed_folder
    output_srt_path = os.path.join(output_folder, f"{os.path.splitext(filename)[0]}.srt")
    
    # Skip if already processed
    if os.path.exists(output_srt_path) and os.path.exists(os.path.join(processed_folder, filename)):
        return {'filename': filename, 'status': 'skipped', 'reason': 'already processed'}
    
    start_time = datetime.now()
    # Make start message prominent and flush immediately
    start_msg = f"\n[{start_time.strftime('%H:%M:%S')}] [Worker {worker_id}] >>> STARTING: {filename}\n"
    print(start_msg, flush=True)
    
    try:
        # Build WhisperX command
        if gpu_access:
            cmd = [
                'whisperx', input_path,
                '--model', 'large-v2',
                '--align_model', 'WAV2VEC2_ASR_LARGE_LV60K_960H',
                '--chunk_size', '4',
                '--language', 'en',
                '--output_format', 'srt',
                '--output_dir', output_folder
            ]
        else:
            cmd = [
                'whisperx', input_path,
                '--model', 'large-v2',
                '--align_model', 'WAV2VEC2_ASR_LARGE_LV60K_960H',
                '--chunk_size', '4',
                '--language', 'en',
                '--output_format', 'srt',
                '--compute_type', 'int8',
                '--output_dir', output_folder
            ]
        
        # Run WhisperX (suppress stdout to reduce noise, but keep stderr for errors)
        # This prevents WhisperX's verbose transcript output from burying our status messages
        result = subprocess.run(
            cmd,
            stdout=subprocess.DEVNULL,  # Suppress WhisperX's verbose output
            stderr=subprocess.PIPE,     # Keep stderr for error messages
            text=True,
            check=False
        )
        
        # Only show stderr if there was an error
        if result.returncode != 0 and result.stderr:
            print(f"[{datetime.now().strftime('%H:%M:%S')}] [Worker {worker_id}] ⚠ WhisperX warning/error: {result.stderr[:300]}", flush=True)
        
        # SAFETY CHECK: Verify SRT file is complete before moving video
        # Wait a moment to ensure file is fully written (WhisperX might still be flushing)
        max_wait = 5  # Maximum seconds to wait for file to stabilize
        wait_interval = 0.5  # Check every 0.5 seconds
        srt_ready = False
        
        for attempt in range(int(max_wait / wait_interval)):
            if os.path.exists(output_srt_path):
                # Check file size (must be > 0 bytes)
                file_size = os.path.getsize(output_srt_path)
                if file_size > 0:
                    # Check if file is still being written (compare modification times)
                    stat1 = os.stat(output_srt_path)
                    time.sleep(wait_interval)
                    stat2 = os.stat(output_srt_path)
                    
                    # If file size and mtime haven't changed, it's complete
                    if stat1.st_size == stat2.st_size and stat1.st_mtime == stat2.st_mtime:
                        # Final validation: try to read first few lines to ensure it's a valid SRT
                        try:
                            with open(output_srt_path, 'r', encoding='utf-8') as f:
                                first_line = f.readline().strip()
                                # SRT files typically start with "1" or have subtitle index
                                if first_line and (first_line.isdigit() or len(first_line) > 0):
                                    srt_ready = True
                                    print(f"[{datetime.now().strftime('%H:%M:%S')}] [Worker {worker_id}] ✓ SRT file validated: {file_size} bytes", flush=True)
                                    break
                        except Exception as e:
                            print(f"[{datetime.now().strftime('%H:%M:%S')}] [Worker {worker_id}] ⚠ SRT file read error: {e}", flush=True)
            time.sleep(wait_interval)
        
        if not srt_ready:
            error_msg = f"SRT file not created or incomplete"
            if os.path.exists(output_srt_path):
                file_size = os.path.getsize(output_srt_path)
                error_msg = f"SRT file exists but appears incomplete or invalid (size: {file_size} bytes)"
            return {
                'filename': filename,
                'status': 'failed',
                'reason': error_msg,
                'error': result.stderr[:200] if result.stderr else 'Unknown error'
            }
        
        # SAFETY: Only move video AFTER confirming SRT is complete
        processed_path = os.path.join(processed_folder, filename)
        if os.path.exists(input_path):
            # Double-check SRT still exists before moving (paranoia check)
            if not os.path.exists(output_srt_path):
                return {
                    'filename': filename,
                    'status': 'failed',
                    'reason': 'SRT file disappeared before move (race condition?)',
                    'error': 'File validation failed'
                }
            shutil.move(input_path, processed_path)
            print(f"[{datetime.now().strftime('%H:%M:%S')}] [Worker {worker_id}] ✓ Video moved to processed folder", flush=True)
        
        elapsed = (datetime.now() - start_time).total_seconds()
        # Make completion message very prominent and flush immediately
        completion_msg = f"\n{'='*80}\n[{datetime.now().strftime('%H:%M:%S')}] [Worker {worker_id}] ✓✓✓ COMPLETED: {filename} ({elapsed:.1f}s)\n{'='*80}\n"
        print(completion_msg, flush=True)
        
        return {
            'filename': filename,
            'status': 'success',
            'elapsed': elapsed
        }
        
    except Exception as e:
        elapsed = (datetime.now() - start_time).total_seconds()
        error_msg = f"\n{'!'*80}\n[{datetime.now().strftime('%H:%M:%S')}] [Worker {worker_id}] ✗✗✗ FAILED: {filename} - {str(e)}\n{'!'*80}\n"
        print(error_msg, flush=True)
        return {
            'filename': filename,
            'status': 'error',
            'reason': str(e),
            'elapsed': elapsed
        }

# Collect all video files to process
video_files = [
    f for f in os.listdir(to_process_folder)
    if f.endswith(('.mp4', '.mkv', '.avi', '.mov', '.flv', '.wmv', '.m4v'))
]

# Remove duplicates (safety check)
video_files = list(dict.fromkeys(video_files))  # Preserves order, removes duplicates

if not video_files:
    print("No video files found in 'To be processed' folder.")
else:
    print(f"Found {len(video_files)} unique video file(s). Processing with {NUM_WORKERS} parallel workers...")
    print(f"GPU Access: {gpu_access}")
    print(f"\nVideos to process:")
    for i, vid in enumerate(video_files, 1):
        print(f"  {i}. {vid}")
    print()
    
    # Process videos in parallel
    results = []
    active_workers = {}  # Track which videos are currently being processed
    
    print(f"{'='*80}")
    print(f"⚠️  IMPORTANT: NUM_WORKERS = {NUM_WORKERS}")
    print(f"⚠️  Only run this cell ONCE. Running it again will create ADDITIONAL workers!")
    print(f"⚠️  If you see workers from different ThreadPoolExecutor instances, you ran this cell multiple times.")
    print(f"{'='*80}\n")
    
    with ThreadPoolExecutor(max_workers=NUM_WORKERS) as executor:
        # Submit all tasks - each filename gets its own unique task
        future_to_file = {}
        for filename in video_files:
            future = executor.submit(process_video, filename)
            future_to_file[future] = filename
            active_workers[filename] = 'queued'
        
        # Process completed tasks as they finish
        for future in as_completed(future_to_file):
            filename = future_to_file[future]
            result = future.result()
            results.append(result)
            if filename in active_workers:
                del active_workers[filename]
    
    # Print summary
    print("\n" + "="*60)
    print("PROCESSING SUMMARY")
    print("="*60)
    successful = [r for r in results if r['status'] == 'success']
    failed = [r for r in results if r['status'] in ['failed', 'error']]
    skipped = [r for r in results if r['status'] == 'skipped']
    
    print(f"✓ Successful: {len(successful)}")
    print(f"✗ Failed: {len(failed)}")
    print(f"⊘ Skipped: {len(skipped)}")
    
    if successful:
        avg_time = sum(r.get('elapsed', 0) for r in successful) / len(successful)
        total_time = sum(r.get('elapsed', 0) for r in successful)
        print(f"\nAverage processing time: {avg_time:.1f}s per video")
        print(f"Total processing time: {total_time:.1f}s")
        if NUM_WORKERS > 1:
            estimated_sequential_time = total_time
            actual_time = max(r.get('elapsed', 0) for r in successful) if successful else 0
            speedup = estimated_sequential_time / actual_time if actual_time > 0 else 1
            print(f"Estimated speedup: {speedup:.1f}x (with {NUM_WORKERS} workers)")
    
    if failed:
        print(f"\nFailed files:")
        for r in failed:
            print(f"  - {r['filename']}: {r.get('reason', 'Unknown error')}")
    
    print("\nProcessing complete.")

Streaming output truncated to the last 5000 lines.
Transcript: [1688.881 --> 1692.543]  inside two working days is holding their accounts
Transcript: [1692.543 --> 1693.606]  payable open.
Transcript: [1694.956 --> 1697.639]  of any millimeter.
Transcript: [1697.639 --> 1701.453]  second after month end. Many of them may be even closing
Transcript: [1701.453 --> 1704.507]  at noon on the last day or even earlier.
Transcript: [1705.486 --> 1706.633]  I promise all of you.
Transcript: [1707.005 --> 1710.447]  You could, if you wanted to find the perfect number,
Transcript: [1710.447 --> 1712.759]  and you left your accounts payable open.
Transcript: [1713.35 --> 1717.163]  for say three weeks, you still wouldn't get the right numbers in.
Transcript: [1717.163 --> 1720.167]  there still be unrecorded liabilities and so forth.
Transcript: [1721.517 --> 1724.352]  So let's close AP.
Transcript: [1726.495 --> 1727.373]  for March.
Transcript: [1728.183 --> 1732.182]  on the last working day 

# pyPreservica

## Install pyPreservica

In [ ]:
# pyPreservica is a Python library for interacting with the Preservica digital preservation platform.
# !pip install pypreservica

## Collect credentials for Preservica

In [ ]:
# User is prompted to enter their Preservica credentials, which are necessary for uploading files.
# from getpass import getpass

# USERNAME = input("Enter your USERNAME: ")
# PASSWORD = getpass("Enter your PASSWORD: ")
# TENANT = input("Enter your TENANT: ") # icaew
# SERVER = input("Enter your SERVER: ") # eu.preservica.com

## Define the Preservica folder ID where the files will be uploaded


In [ ]:
# preservica_folder_id = "de6831c5-9583-4f51-8dd8-6eadf7a56c52"

## Import necessary classes from pyPreservica

In [ ]:
# from pyPreservica import UploadAPI, complex_asset_package
# import os

## Define the path to the 'Processed' folder

In [ ]:
# This folder contains the processed videos and their corresponding subtitle files.
# processed_folder = '/content/drive/My Drive/Media for transcription/Processed'

## Initialize Preservica client

In [ ]:
# This creates an instance of the Preservica client, which will be used to upload files.
# client = UploadAPI(username=USERNAME, password=PASSWORD,
#                    tenant=TENANT, server=SERVER)

## Upload processed files to Preservica

In [ ]:
# for filename in os.listdir(processed_folder):
#     if filename.endswith(('.mp4', '.mkv', '.avi', '.mov', '.flv')):  # Add more extensions as needed
#         video_path = os.path.join(processed_folder, filename)
#         srt_path = os.path.join(processed_folder, f"{os.path.splitext(filename)[0]}.srt")

#         # Check if the corresponding subtitle file exists
#         if os.path.exists(srt_path):
#             files_to_upload = [video_path, srt_path]
#         else:
#             files_to_upload = [video_path]

#         print(f"Uploading files for {filename} to Preservica...")

#         # Create a complex asset package and upload it
#         package = complex_asset_package(files_to_upload, parent_folder=preservica_folder_id, Description="")
#         client.upload_zip_package(package)

#         print(f"Uploaded {filename} and corresponding subtitles to Preservica.")

#         # Move the files to the "Processed and Ingested" folder after upload
#         ingested_video_path = os.path.join(processed_and_ingested_folder, filename)
#         shutil.move(video_path, ingested_video_path)

#         if os.path.exists(srt_path):
#             ingested_srt_path = os.path.join(processed_and_ingested_folder, os.path.basename(srt_path))
#             shutil.move(srt_path, ingested_srt_path)

# print("All files uploaded to Preservica and moved to the 'Processed and Ingested' folder.")